In [1]:
# Librerías
import requests
import selectolax
from selectolax.parser import HTMLParser
import pandas as pd
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo

# Funciones del proyecto
import scrapping_functions
from scrapping_functions import get_json_from_url, save_json, parse_json_to_model
import html_utils
from html_utils import esperar, obtener_max_page, extraer_productos, save_df_as_csv

##### Categorias

In [2]:
response = requests.get("https://www.superseis.com.py/default.aspx")
html = response.text
tree = HTMLParser(html)

In [3]:
data = []

# Buscar todos los nodos de nivel 1
for lvl1_li in tree.css("li.level1"):
    lvl1_a = lvl1_li.css_first("a")
    if not lvl1_a:
        continue
    lvl1_name = lvl1_a.text(strip=True)

    # Dentro de este <li>, buscar los hijos de nivel 2
    for lvl2_li in lvl1_li.css("ul > li.level2"):
        lvl2_a = lvl2_li.css_first("a")
        if not lvl2_a:
            continue
        lvl2_name = lvl2_a.text(strip=True)

        # Dentro del lvl2, buscar los hijos de nivel 3 (con enlaces)
        for lvl3_li in lvl2_li.css("ul > li.level3"):
            lvl3_a = lvl3_li.css_first("a[href]")
            if not lvl3_a:
                continue
            lvl3_name = lvl3_a.text(strip=True)
            lvl3_url = lvl3_a.attributes.get("href")

            data.append({
                "categoria_nivel_1": lvl1_name,
                "categoria_nivel_2": lvl2_name,
                "categoria_nivel_3": lvl3_name,
                "url": lvl3_url,
                "category_slug": f"{lvl1_name}/{lvl2_name}/{lvl3_name}".replace(" ", "_")
            })

In [4]:
df_categorias = pd.DataFrame(data)
df_categorias.nunique()

categoria_nivel_1     19
categoria_nivel_2     80
categoria_nivel_3    304
url                  321
category_slug        321
dtype: int64

In [5]:
df_categorias.head()

,categoria_nivel_1,categoria_nivel_2,categoria_nivel_3,url,category_slug
0,Almacén,Aderezos/Condimentos,Aceites,https://www.superseis.com.py/category/3-almace...,Almacén/Aderezos/Condimentos/Aceites
1,Almacén,Aderezos/Condimentos,Aderezos/Salsas,https://www.superseis.com.py/category/4-almace...,Almacén/Aderezos/Condimentos/Aderezos/Salsas
2,Almacén,Aderezos/Condimentos,Especias,https://www.superseis.com.py/category/5-almace...,Almacén/Aderezos/Condimentos/Especias
3,Almacén,Aderezos/Condimentos,Ketchup,https://www.superseis.com.py/category/6-almace...,Almacén/Aderezos/Condimentos/Ketchup
4,Almacén,Aderezos/Condimentos,Mayonesa,https://www.superseis.com.py/category/7-almace...,Almacén/Aderezos/Condimentos/Mayonesa


In [10]:
save_df_as_csv(df_categorias, "/workspaces/tesis-ivan-gennaro/scripts/bronze/outputs", "s6_categories")

Dataframe guardado en ubicación /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/s6_categories


# Productos

In [9]:
df_categorias = df_categorias[0:1]
df_categorias

,categoria_nivel_1,categoria_nivel_2,categoria_nivel_3,url,category_slug
0,Almacén,Aderezos/Condimentos,Aceites,https://www.superseis.com.py/category/3-almace...,Almacén/Aderezos/Condimentos/Aceites


In [10]:
INGESTION_TIME = datetime.now(ZoneInfo("America/Asuncion"))
SUPERMERCADO = "Super Seis"
productos_final = []

# 🔁 Iterar sobre cada categoría (nivel 3) con contexto
for _, row in df_categorias.iterrows():
    categoria_url = row["url"]
    category_slug = row["category_slug"]

    print(f"\n🔎 Scrapeando categoría: {category_slug}")
    esperar()
    
    response = requests.get(categoria_url)
    tree = HTMLParser(response.text)
    max_page = obtener_max_page(tree)
    print(f"📄 Total de páginas: {max_page}")

    for page in range(1, max_page + 1):
        page_url = f"{categoria_url}?pageindex={page}"
        print(f"➡️ Página {page}: {page_url}")
        esperar()

        resp = requests.get(page_url)
        html_tree = HTMLParser(resp.text)

        contexto = {
            "category_slug": category_slug,
            "ingestion_time": INGESTION_TIME,
            "supermercado": SUPERMERCADO
        }

    productos_pagina = extraer_productos(html_tree, contexto)
    productos_final.extend(productos_pagina)



🔎 Scrapeando categoría: Almacén/Aderezos/Condimentos/Aceites
📄 Total de páginas: 2
➡️ Página 1: https://www.superseis.com.py/category/3-almacen-aderezoscondimentos-aceites.aspx?pageindex=1
➡️ Página 2: https://www.superseis.com.py/category/3-almacen-aderezoscondimentos-aceites.aspx?pageindex=2


In [11]:
df = pd.DataFrame(productos_final)
df.head(10)

,titulo,marca,precio,category_slug,ingestion_time,supermercado
0,ACEITE MEZCLA DE SOJA Y CANOLA EN BOTELLA REIN...,,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
1,ACEITE D/COCO SIN SABOR COPRA 200ML FCO,,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
2,ACEITE DE GIRASOL EN BOTELLA PRIMOR 900 ML,,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
3,ACEITE DE OLIVA 125 ML LA ESPAÑOLA FRASCO,,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
4,ACEITE DE SOJA EN BOTELLA BIANCA 1.5 LITROS,,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
5,ACEITE DE OLIVA EXTRA VIRGEN YBARRA 250ML,MARCA EXCLUSIVA,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
6,ACEITE DE OLIVA EXTRA VIRGEN YBARRA 500ML,MARCA EXCLUSIVA,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
7,ACEITE DE OLIVA EXTRA VIRGEN YBARRA 750ML,MARCA EXCLUSIVA,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
8,ACEITE DE OLIVA EN BOTELLA YBARRA 500 ML,MARCA EXCLUSIVA,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis
9,ACEITE DE OLIVA EN BOTELLA YBARRA 750 ML,MARCA EXCLUSIVA,Un.,Almacén/Aderezos/Condimentos/Aceites,2025-07-20 22:59:45.235247-03:00,Super Seis


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype                           
---  ------          --------------  -----                           
 0   titulo          25 non-null     object                          
 1   marca           25 non-null     object                          
 2   precio          25 non-null     object                          
 3   category_slug   25 non-null     object                          
 4   ingestion_time  25 non-null     datetime64[ns, America/Asuncion]
 5   supermercado    25 non-null     object                          
dtypes: datetime64[ns, America/Asuncion](1), object(5)
memory usage: 1.3+ KB
